สำรวจแหล่งข้อมูล ของ PEA: https://docs.google.com/spreadsheets/d/1Vw5Z4qYNjKXsFCeb2b1-VHhJV_tZGhK0wURNVI3DnYg/edit?gid=0#gid=0
- 1.1.1.1 ข้อมูลแผนที่ในปทุมธานี
- 1.1.1.5 ข้อมูลหน่วยงานรัฐในปทุมธานี
- 1.1.1.6 ข้อมูลหน่วยงานเอกชนในปทุมธานี พื้นที่ที่มีโรงงาน หรือธุรกิจขนาดใหญ่
- 1.1.1.7 ข้อมูลความหนาแน่นประชากรในปทุมธานี

In [1]:
# pip install osmnx geopandas folium matplotlib

# **1.1.1.1 ข้อมูลแผนที่ในปทุมธานี**

In [54]:
import requests
import folium
import warnings
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import osmnx as ox
import warnings

warnings.filterwarnings('ignore')

### -- **Landing** --

In [55]:
place = "Pathum Thani, Thailand"
pathum_poly = ox.geocode_to_gdf(place).geometry.iloc[0]

# --- ดึงข้อมูล ตำบล (Admin Level 8) ---
gdf_tam = ox.features_from_polygon(pathum_poly, tags={"boundary": "administrative", "admin_level": "8"})
gdf_tam.info()
gdf_tam.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 393 entries, ('relation', 92277) to ('way', 1396764064)
Columns: 139 entries, geometry to website
dtypes: geometry(1), object(138)
memory usage: 468.4+ KB


geometry  \
element  id                                                           
relation 92277    POLYGON ((100.32788 13.80418, 100.33185 13.803...   
         1639287  POLYGON ((100.5589 13.91448, 100.56004 13.9194...   
         1908767  POLYGON ((100.74628 13.21895, 100.84831 13.477...   
         1908787  POLYGON ((100.91381 14.1824, 100.91396 14.2096...   
         1908796  POLYGON ((100.26378 13.99934, 100.26408 13.999...   

                 admin_level is_in:country               name  \
element  id                                                     
relation 92277             4      Thailand      กรุงเทพมหานคร   
         1639287           6           NaN        เขตดอนเมือง   
         1908767           4           NaN  จังหวัดฉะเชิงเทรา   
         1908787           4           NaN     จังหวัดนครนายก   
         1908796           4           NaN     จังหวัดนนทบุรี   

                                name:en       name:fr     name:ja  \
element  id                                                         
relation 92277                  Bangkok       Bangkok        バンコク   
         1639287    Don Mueang District           NaN         NaN   
         1908767  Chachoengsao Province  Chachoengsao   チャチューンサオ県   
         1908787  Nakhon Nayok Province  Nakhon Nayok  ナコーンナーヨック県   
         1908796    Nonthaburi Province    Nonthaburi     ノンタブリー県   

                      name:ru            name:th population  ... old_name:th  \
element  id                                                  ...               
relation 92277        Бангкок      กรุงเทพมหานคร        NaN  ...      บางกอก   
         1639287          NaN        เขตดอนเมือง     168896  ...         NaN   
         1908767    Чаченгсау  จังหวัดฉะเชิงเทรา        NaN  ...         NaN   
         1908787  Накхоннайок     จังหวัดนครนายก        NaN  ...         NaN   
         1908796   Нонтхабури     จังหวัดนนทบุรี        NaN  ...         NaN   

                 postal_code             alt_name:pl               name:ace  \
element  id                                                                   
relation 92277           NaN                     NaN                    NaN   
         1639287       10210                     NaN                    NaN   
         1908767         NaN  Prowincja Chachoengsao  Propinsi Chachoengsao   
         1908787         NaN  Prowincja Nakhon Nayok  Propinsi Nakhon Nayok   
         1908796         NaN    Prowincja Nonthaburi    Propinsi Nonthaburi   

                         name:cdo         name:nan                 name:no  \
element  id                                                                  
relation 92277                NaN              NaN                     NaN   
         1639287              NaN              NaN                     NaN   
         1908767  Chachoengsao Hū  Chachoengsao Hú            Chachoengsao   
         1908787  Nakhon Nayok Hū  Nakhon Nayok Hú  Nakhon Nayok (provins)   
         1908796    Nonthaburi Hū    Nonthaburi Hú              Nonthaburi   

                 alt_name:ja wikipedia:th website  
element  id                                        
relation 92277           NaN          NaN     NaN  
         1639287         NaN          NaN     NaN  
         1908767         NaN          NaN     NaN  
         1908787         NaN          NaN     NaN  
         1908796         NaN          NaN     NaN  

[5 rows x 139 columns]

### **Staging**

In [72]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. STAGING PHASE: Data Extraction & Cleansing


print("--- [Staging] เริ่มต้นดึงข้อมูลขอบเขตปทุมธานี ---")
place = "Pathum Thani, Thailand"
pathum_poly = ox.geocode_to_gdf(place).geometry.iloc[0]

def extract_and_clean_admin(level, exclude_keywords):
    print(f"กำลังดึงข้อมูล Admin Level {level}...")
    gdf = ox.features_from_polygon(pathum_poly, tags={"boundary": "administrative", "admin_level": str(level)})
    gdf = gdf[gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])]
    gdf = gdf[gdf.geometry.centroid.within(pathum_poly)]
    gdf = gdf.reset_index()
    if 'osmid' not in gdf.columns:
        gdf['osmid'] = range(len(gdf))
    gdf['name_th'] = gdf['name:th'].fillna(gdf.get('name', 'ไม่ระบุ')).fillna('ไม่ระบุ')
    gdf = gdf[~gdf['name_th'].str.contains(exclude_keywords, na=False)]
    if level == 8:
        gdf['area_name'] = gdf['name_th'].str.replace('ตำบล', '').str.replace('ต.', '').str.strip()
    elif level == 6:
        gdf['area_name'] = gdf['name_th'].str.replace('อำเภอ', '').str.replace('อ.', '').str.strip()
    gdf['area_calc'] = gdf.geometry.area 
    if 'element_type' in gdf.columns:
        gdf['is_rel'] = gdf['element_type'] == 'relation'
        gdf = gdf.sort_values(by=['is_rel', 'area_calc'], ascending=[False, False])
    else:
        gdf = gdf.sort_values(by=['area_calc'], ascending=[False])
    gdf = gdf.drop_duplicates(subset='area_name')
    gdf = gdf[gdf['area_name'] != 'ไม่ระบุ']
    gdf['area_sqkm'] = round(gdf.to_crs(epsg=32647).geometry.area / 1e6, 2)
    gdf['center_latitude'] = gdf.geometry.centroid.y
    gdf['center_longitude'] = gdf.geometry.centroid.x
    gdf['source_id'] = gdf['osmid'].astype(str) 
    
    return gdf[['source_id', 'area_name', 'geometry', 'area_sqkm', 'center_latitude', 'center_longitude']]

# รัน Staging สำหรับอำเภอและตำบล
df_amp_stg = extract_and_clean_admin(6, 'เทศบาล|อบต|องค์การ|จังหวัด')
df_tam_stg = extract_and_clean_admin(8, 'เทศบาล|อบต|องค์การ|จังหวัด|อำเภอ|ตำบล')

df_tam_stg.info()
df_tam_stg.head()

--- [Staging] เริ่มต้นดึงข้อมูลขอบเขตปทุมธานี ---
กำลังดึงข้อมูล Admin Level 6...
กำลังดึงข้อมูล Admin Level 8...
<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 0 entries
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   source_id         0 non-null      object  
 1   area_name         0 non-null      object  
 2   geometry          0 non-null      geometry
 3   area_sqkm         0 non-null      float64 
 4   center_latitude   0 non-null      float64 
 5   center_longitude  0 non-null      float64 
dtypes: float64(3), geometry(1), object(2)
memory usage: 0.0+ bytes


,source_id,area_name,geometry,area_sqkm,center_latitude,center_longitude


In [73]:
df_amp_stg.info()
df_amp_stg.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 67 entries, 5 to 18
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   source_id         67 non-null     object  
 1   area_name         67 non-null     object  
 2   geometry          67 non-null     geometry
 3   area_sqkm         67 non-null     float64 
 4   center_latitude   67 non-null     float64 
 5   center_longitude  67 non-null     float64 
dtypes: float64(3), geometry(1), object(2)
memory usage: 3.7+ KB


,source_id,area_name,geometry,area_sqkm,center_latitude,center_longitude
5,5,หนองเสือ,"POLYGON ((100.75518 14.21112, 100.76637 14.216...",339.73,14.161772,100.838181
1,1,คลองหลวง,"POLYGON ((100.58221 14.12866, 100.58255 14.128...",304.18,14.095031,100.676509
2,2,ลำลูกกา,"POLYGON ((100.60511 13.96579, 100.60626 13.966...",303.65,13.978383,100.790889
3,3,ลาดหลุมแก้ว,"POLYGON ((100.34159 14.10102, 100.34162 14.102...",193.63,14.048224,100.406910
4,4,เมืองปทุมธานี,"POLYGON ((100.44987 14.01565, 100.45224 14.019...",143.11,13.993082,100.535143


In [65]:
# # ==========================================
# # 1. ดึงข้อมูลและ Clean ข้อมูล (ตำบล และ อำเภอ)

# print("กำลังเตรียมข้อมูลขอบเขต...")
# gdf_tam = gdf_tam[gdf_tam.geometry.type.isin(['Polygon', 'MultiPolygon'])]
# gdf_tam = gdf_tam[gdf_tam.geometry.centroid.within(pathum_poly)]
# gdf_tam = gdf_tam.reset_index()

# gdf_tam['name_th'] = gdf_tam['name:th'].fillna(gdf_tam.get('name', 'ไม่ระบุ')).fillna('ไม่ระบุ')
# gdf_tam['name_en'] = gdf_tam['name:en'].fillna(gdf_tam.get('name', 'Unknown'))

# # Clean แบบ Hardcode ตามที่คุณทำไว้
# gdf_tam = gdf_tam[~gdf_tam['name_th'].str.contains('เทศบาล|อบต|องค์การ|จังหวัด|อำเภอ', na=False)]
# gdf_tam['tam_th'] = gdf_tam['name_th'].str.replace('ตำบล', '').str.replace('ต.', '').str.strip() # เปลี่ยนชื่อเป็น tam_th เลยเพื่อใช้ใน Folium
# gdf_tam['tam_en'] = gdf_tam['name_en']
# gdf_tam['area_calc'] = gdf_tam.geometry.area

# if 'element_type' in gdf_tam.columns:
#     gdf_tam['is_rel'] = gdf_tam['element_type'] == 'relation'
#     gdf_tam = gdf_tam.sort_values(by=['is_rel', 'area_calc'], ascending=[False, False])
# else:
#     gdf_tam = gdf_tam.sort_values(by=['area_calc'], ascending=[False])

# gdf_tam = gdf_tam.drop_duplicates(subset='tam_th')
# gdf_tam = gdf_tam[gdf_tam['tam_th'] != 'ไม่ระบุ']


# # --- ดึงข้อมูล อำเภอ (Admin Level 6) เพื่อเอามาใส่ Tooltip ---
# gdf_amp = ox.features_from_polygon(pathum_poly, tags={"boundary": "administrative", "admin_level": "6"})
# gdf_amp = gdf_amp[gdf_amp.geometry.type.isin(['Polygon', 'MultiPolygon'])]
# gdf_amp = gdf_amp[gdf_amp.geometry.centroid.within(pathum_poly)]
# gdf_amp = gdf_amp.reset_index()
# gdf_amp['amp_th'] = gdf_amp['name:th'].fillna(gdf_amp.get('name', 'ไม่ระบุ')).fillna('ไม่ระบุ')
# gdf_amp['amp_th'] = gdf_amp['amp_th'].str.replace('อำเภอ', '').str.replace('อ.', '').str.strip()
# gdf_amp = gdf_amp.drop_duplicates(subset='amp_th')


# # ==========================================
# # 2. ผูกข้อมูลตำบลเข้ากับอำเภอ และคำนวณพื้นที่

# print("กำลังผูกข้อมูลตำบลกับอำเภอ...")
# tam_centroid = gdf_tam[['geometry', 'tam_th']].copy()
# tam_centroid.geometry = tam_centroid.geometry.centroid
# joined = gpd.sjoin(tam_centroid, gdf_amp[['geometry', 'amp_th']], how='left', predicate='intersects')
# joined = joined.drop_duplicates(subset=['tam_th'])
# gdf_tam = gdf_tam.merge(joined[['tam_th', 'amp_th']], on='tam_th', how='left')
# gdf_tam['area_sqkm'] = round(gdf_tam.to_crs(epsg=32647).geometry.area / 1e6, 2)


# # ==========================================
# # 3. สร้างแผนที่ FOLIUM

# print(f"เตรียมข้อมูลสำเร็จ! นำไปพล็อตทั้งหมด {len(gdf_tam)} ตำบล")
# unique_tams = gdf_tam['tam_th'].unique()
# colors = plt.cm.get_cmap('tab20', len(unique_tams))
# color_dict = {name: mcolors.to_hex(colors(i)) for i, name in enumerate(unique_tams)}
# m = folium.Map(
#     location=[pathum_poly.centroid.y, pathum_poly.centroid.x], 
#     zoom_start=11,
#     tiles='cartodbpositron' 
# )
# folium.GeoJson(
#     gdf_tam,
#     name='ขอบเขตตำบลในปทุมธานี',
#     style_function=lambda x: {
#         'fillColor': color_dict.get(x['properties'].get('tam_th', ''), '#ffffff'),
#         'color': 'black',
#         'weight': 0.5,          
#         'fillOpacity': 0.6      
#     },
#     highlight_function=lambda x: {
#         'weight': 3, 
#         'fillOpacity': 0.9,
#         'color': 'red'         
#     },
#     tooltip=folium.GeoJsonTooltip(
#         fields=['tam_th', 'tam_en', 'amp_th', 'area_sqkm'],
#         aliases=['ตำบล: ', 'Sub-district: ', 'อำเภอ: ', 'พื้นที่ (ตร.กม.): '],
#         localize=True,
#         sticky=False,
#         labels=True,
#         style="""
#             background-color: #F0EFEF;
#             border: 2px solid black;
#             border-radius: 3px;
#             box-shadow: 3px;
#             font-family: 'Tahoma', sans-serif;
#         """
#     )
# ).add_to(m)
# folium.LayerControl().add_to(m)
# m

### -- **Integration** --

In [75]:
# ===========================================================
# 2. INTEGRATION PHASE: Hierarchy Mapping & Schema Formatting

print("--- [Integration] เริ่มประกอบร่างข้อมูลลง Schema ---")
df_amp_int = df_amp_stg.copy()
df_amp_int['area_id'] = ['AMP_' + str(i).zfill(3) for i in range(1, len(df_amp_int) + 1)]
df_amp_int['location_type'] = 'Amphoe'
df_amp_int['parent_area_id'] = 'PROV_PATHUM' 
df_amp_int['amphoe_name'] = df_amp_int['area_name']
df_amp_int['province_name'] = 'ปทุมธานี'
tam_centroid = df_tam_stg[['geometry', 'area_name']].copy()
tam_centroid.geometry = tam_centroid.geometry.centroid
joined = gpd.sjoin(tam_centroid, df_amp_int[['geometry', 'area_id', 'area_name']], how='left', predicate='intersects')
joined = joined.rename(columns={'area_id': 'parent_area_id', 'area_name_right': 'amphoe_name'})
joined = joined.drop_duplicates(subset=['area_name_left'])

df_tam_int = df_tam_stg.copy()
df_tam_int = df_tam_int.merge(joined[['area_name_left', 'parent_area_id', 'amphoe_name']], left_on='area_name', right_on='area_name_left', how='left')

df_tam_int['area_id'] = ['TAM_' + str(i).zfill(3) for i in range(1, len(df_tam_int) + 1)]
df_tam_int['location_type'] = 'Tambon'
df_tam_int['province_name'] = 'ปทุมธานี'
df_amp_int['geometry_wkt'] = df_amp_int['geometry'].apply(lambda x: x.wkt)
df_tam_int['geometry_wkt'] = df_tam_int['geometry'].apply(lambda x: x.wkt)
MapData = pd.concat([df_amp_int, df_tam_int], ignore_index=True)

final_schema_cols = [
    'area_id', 
    'source_id', 
    'area_name', 
    'location_type', 
    'parent_area_id', 
    'amphoe_name', 
    'province_name', 
    'center_latitude', 
    'center_longitude', 
    'geometry_wkt', 
    'area_sqkm'
]

MapData_Final = MapData[final_schema_cols]
MapData_Final.info()
MapData_Final.head()

--- [Integration] เริ่มประกอบร่างข้อมูลลง Schema ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   area_id           67 non-null     object 
 1   source_id         67 non-null     object 
 2   area_name         67 non-null     object 
 3   location_type     67 non-null     object 
 4   parent_area_id    67 non-null     object 
 5   amphoe_name       67 non-null     object 
 6   province_name     67 non-null     object 
 7   center_latitude   67 non-null     float64
 8   center_longitude  67 non-null     float64
 9   geometry_wkt      67 non-null     object 
 10  area_sqkm         67 non-null     float64
dtypes: float64(3), object(8)
memory usage: 5.9+ KB


,area_id,source_id,area_name,location_type,parent_area_id,amphoe_name,province_name,center_latitude,center_longitude,geometry_wkt,area_sqkm
0,AMP_001,5,หนองเสือ,Amphoe,PROV_PATHUM,หนองเสือ,ปทุมธานี,14.161772,100.838181,"POLYGON ((100.7551769 14.2111236, 100.7663713 ...",339.73
1,AMP_002,1,คลองหลวง,Amphoe,PROV_PATHUM,คลองหลวง,ปทุมธานี,14.095031,100.676509,"POLYGON ((100.5822118 14.1286615, 100.5825547 ...",304.18
2,AMP_003,2,ลำลูกกา,Amphoe,PROV_PATHUM,ลำลูกกา,ปทุมธานี,13.978383,100.790889,"POLYGON ((100.6051065 13.965786, 100.6062589 1...",303.65
3,AMP_004,3,ลาดหลุมแก้ว,Amphoe,PROV_PATHUM,ลาดหลุมแก้ว,ปทุมธานี,14.048224,100.406910,"POLYGON ((100.3415922 14.101018, 100.3416187 1...",193.63
4,AMP_005,4,เมืองปทุมธานี,Amphoe,PROV_PATHUM,เมืองปทุมธานี,ปทุมธานี,13.993082,100.535143,"POLYGON ((100.4498667 14.0156542, 100.4522446 ...",143.11


# **1.1.1.5 ข้อมูลหน่วยงานรัฐในปทุมธานี**

In [5]:
import requests
import pandas as pd

### -- **Landing** --

In [6]:
URL = 'https://catalog.dopa.go.th/dataset/c0cddb04-4b6e-40bc-afb9-ea72d28324bb/resource/c6ed87eb-a8f4-4833-8901-2170e6c093b1/download/.json'
req = requests.get(URL)
df_department = pd.DataFrame(req.json())
# df_department = df_department[df_department['oct_side01_name'].str.contains('องค์การบริหาร')]
df_department.info()
df_department.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2980 entries, 0 to 2979
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   pcode                2980 non-null   object
 1   pname                2980 non-null   object
 2   acode                2980 non-null   object
 3   aname                2980 non-null   object
 4   tcode                2980 non-null   object
 5   tname                2980 non-null   object
 6   mcode                2980 non-null   object
 7   mname                2980 non-null   object
 8   gov_office_id        2980 non-null   object
 9   gov_sub_office_id    2980 non-null   object
 10  oct_side01_name      2941 non-null   object
 11  oct_side01_lat       2941 non-null   object
 12  oct_side01_lon       2941 non-null   object
 13  month_id             2980 non-null   object
 14  budget_id            2980 non-null   object
 15  oct_side01_gov_name  36 non-null     object
 16  oct_si

,pcode,pname,acode,aname,tcode,tname,mcode,mname,gov_office_id,gov_sub_office_id,oct_side01_name,oct_side01_lat,oct_side01_lon,month_id,budget_id,oct_side01_gov_name,oct_side01_pname
0,13,ปทุมธานี,1305,ลาดหลุมแก้ว,13050300,คูบางหลวง,13050303,คลองบางโพธิ์,1,1020,องค์การบริหารส่วนตำบลคูบางหลวง,14.062185836378802,100.46047702431679,11,2564,None,None
1,13,ปทุมธานี,1302,คลองหลวง,13020100,คลองหนึ่ง,13020150,ชุมชนบางขัน,3,3030,วัดบางขัน,14.065239063675945,100.61930134892464,11,2564,None,None
2,13,ปทุมธานี,1305,ลาดหลุมแก้ว,13050300,คูบางหลวง,13050301,คลองบางเตย,2,2110,โรงเรียนวัดจันทาราม,14.070075750195645,100.47903522849083,11,2564,None,None
3,13,ปทุมธานี,1305,ลาดหลุมแก้ว,13050300,คูบางหลวง,13050301,คลองบางเตย,3,3030,วัดจันทาราม,14.071027985190469,100.47882467508316,11,2564,None,None
4,13,ปทุมธานี,1302,คลองหลวง,13020300,คลองสาม,13020309,หมู่ที่ 9,4,4013,,14.079245400211239,100.66492438316345,11,2564,None,None


In [17]:
# https://data.go.th/dataset/health-citizeninfo
df_hospital = pd.read_csv('./input/landing/1.1.1.5/citizeninfo_health_20200314.csv')
df_hospital.info()
df_hospital.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10714 entries, 0 to 10713
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   รหัสหน่วยงาน             10714 non-null  int64  
 1   กระทรวง                  10714 non-null  object 
 2   กรม                      10714 non-null  object 
 3   หน่วยงาน                 10714 non-null  object 
 4   หมุดที่ไม่พบความผิดปกติ  10714 non-null  object 
 5   ที่อยู่จุดบริการ         10622 non-null  object 
 6   ละติจูด                  10622 non-null  float64
 7   ลองติจูด                 10622 non-null  float64
dtypes: float64(2), int64(1), object(5)
memory usage: 669.8+ KB


,รหัสหน่วยงาน,กระทรวง,กรม,หน่วยงาน,หมุดที่ไม่พบความผิดปกติ,ที่อยู่จุดบริการ,ละติจูด,ลองติจูด
0,8899021500001,กระทรวงกลาโหม,กรมแพทย์ทหารบก,โรงพยาบาลค่ายพิชัยดาบหัก,Y,102 หมู่ 8 ต.ท่าเสา อ.เมืองอุตรดิตถ์ จ.อุตรดิต...,17.662624,100.138907
1,8899021200003,กระทรวงกลาโหม,กรมแพทย์ทหารเรือ,โรงพยาบาลสมเด็จพระนางเจ้าสิริกิติ์,N,NaN,NaN,NaN
2,8899021200001,กระทรวงกลาโหม,กรมแพทย์ทหารเรือ,โรงพยาบาลทหารเรือกรุงเทพ,Y,224 ถนนริมทางรถไฟเก่า แขวงบางนา เขตบางนา กรุงเ...,13.670278,100.588009
3,111020400029,กระทรวงกลาโหม,กองทัพบก,กองทัพบก โรงพยาบาลพระมงกุฎเกล้า,Y,แขวงทุ่งพญาไท เขตราชเทวี กรุงเทพมหานคร 10400,13.767327,100.534169
4,113020400030,กระทรวงกลาโหม,กองทัพบก,กองทัพบก โรงพยาบาลอานันทมหิดล,Y,ต.เขาสามยอด อ.เมืองลพบุรี จ.ลพบุรี 15000,14.849659,100.666970


### -- **Staging** --

# **1.1.1.6 ข้อมูลหน่วยงานเอกชนในปทุมธานี พื้นที่ที่มีโรงงาน หรือธุรกิจขนาดใหญ่**

### -- **Landing** --

In [16]:
df_factory = pd.read_csv('./input/landing/1.1.1.6/central.csv')
df_factory.info()
df_factory.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17570 entries, 0 to 17569
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   FID          17570 non-null  int64  
 1   DISPFACREG   17570 non-null  object 
 2   FNAME        17013 non-null  object 
 3   ONAME        17570 non-null  object 
 4   FADDR        17222 non-null  object 
 5   FMOO         16960 non-null  object 
 6   FSOI         7554 non-null   object 
 7   FROAD        13175 non-null  object 
 8   FTUMNAME     17570 non-null  object 
 9   FAMPNAME     17570 non-null  object 
 10  FPROVNAME    17570 non-null  object 
 11  OBJECT       17564 non-null  object 
 12  STATUS       17570 non-null  object 
 13  LAT          17570 non-null  float64
 14  LNG          17570 non-null  float64
 15  Last_update  17570 non-null  object 
dtypes: float64(2), int64(1), object(13)
memory usage: 2.1+ MB


,FID,DISPFACREG,FNAME,ONAME,FADDR,FMOO,FSOI,FROAD,FTUMNAME,FAMPNAME,FPROVNAME,OBJECT,STATUS,LAT,LNG,Last_update
0,40190000925561,3-88(1)-9/56สบ,บริษัท อินฟินิท กรีน จำกัด,บริษัท อินฟินิท กรีน จำกัด,โฉนด31541,1,NaN,บ้านเขาเกตุ-บ้านดง,ชำผักแพว,แก่งคอย,สระบุรี,โรงงานผลิตไฟฟ้าจากพลังงานแสงอาทิตย์ 1.68 เมกะว...,ดำเนินกิจการ,14.556038,101.075985,2023-04-18 21:00:05.000
1,20110001625592,จ3-64(13)-16/59สป,-,นางสมคิด เอี่ยมภิรมย์,290/81,12,NaN,NaN,ในคลองบางปลากด,พระสมุทรเจดีย์,สมุทรปราการ,กลึง เจาะ กัด ไส เจียน หรือเชื่อมโลหะทั่วไป แล...,ดำเนินกิจการ,13.603355,100.540314,2023-04-18 21:00:05.000
2,10260000125334,3-3(4)-1/33นย,บ่อทรายสามชัย,นายสุรชัย สุภาวิมล,185,2,NaN,บ้านนา-แก่งคอย,เขาเพิ่ม,บ้านนา,นครนายก,ดูดทราย,ดำเนินกิจการ,14.347670,101.086340,2023-04-18 21:00:05.000
3,10260000125201,3-34(1)-1/20นย,บริษัท ลิ้มเจริญค้าว้สถุก่อสร้าง จำกัด,บริษัท ลิ้มเจริญค้าวัสดุก่อสร้าง จำกัด,120,1,NaN,NaN,องครักษ์,องครักษ์,นครนายก,ไสไม้และซอยไม้,ดำเนินกิจการ,14.120700,100.999790,2023-04-18 21:00:05.000
4,10260001325156,3-9(1)-13/15นย,ห้างหุ้นส่วนจำกัด ตี่ฮั่วเชียง,ห้างหุ้นส่วนจำกัด ตี่ฮั่วเชียง,22,5,NaN,รังสิต-นครนายก,พระอาจารย์,องครักษ์,นครนายก,สีข้าว กำลังสีสูงสุดของร้านสีข้าว 20 เกวียน/วัน,หยุดชั่วคราว,13.971370,100.961360,2023-04-18 21:00:05.000


### -- **Staging** --

# **1.1.1.7 ข้อมูลความหนาแน่นประชากรในปทุมธานี**

In [1]:
# pip install lxml

In [15]:
import pandas as pd
import requests, io

### -- **Landing** --

In [46]:
cols = [
    'year_month',    # YYMM (ปีเดือนข้อมูล)
    'prov_code',     # CC-CODE (รหัสจังหวัด)
    'prov_name',     # CC-DESC (ชื่อจังหวัด)
    'amp_code',      # RCODE-CODE (รหัสสำนักทะเบียน/อำเภอ)
    'amp_name',      # RCODE-DESC (ชื่อสำนักทะเบียน/อำเภอ)
    'tam_code',      # CCAATT-CODE (รหัสตำบล)
    'tam_name',      # CCAATT-DESC (ชื่อตำบล)
    'moo_code',      # CCAATTMM-CODE (รหัสหมู่บ้าน)
    'moo_name',      # CCAATTMM-DESC (ชื่อหมู่บ้าน)
    'male',          # MALE (จำนวนประชากรชาย)
    'female',        # FEMALE (จำนวนประชากรหญิง)
    'total',         # TOTAL (จำนวนประชากรทั้งหมด)
    'households',    # HOUSE (จำนวนบ้าน)
    'empty_drop'     # (จำเป็นต้องมี! ไว้รับมือกับเครื่องหมาย | ตัวสุดท้ายที่ติดมาหลังสุด)
]

year = 66
URL = f"https://stat.bora.dopa.go.th/new_stat/file/{year}/stat_t{year}.txt"
req = requests.get(URL)

# 1. อ่านข้อมูลเข้า Pandas
df_dopa = pd.read_csv(
    io.StringIO(req.text), 
    sep='|', 
    header=None, 
    names=cols,
    thousands=',',              
    encoding='utf-8-sig',         
    dtype={'prov_code': str, 'amp_code': str, 'tam_code': str} 
)
df_dopa.info()
df_dopa.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11375 entries, 0 to 11374
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   year_month  11375 non-null  int64  
 1   prov_code   11375 non-null  object 
 2   prov_name   11375 non-null  object 
 3   amp_code    11375 non-null  object 
 4   amp_name    11375 non-null  object 
 5   tam_code    11375 non-null  object 
 6   tam_name    11375 non-null  object 
 7   moo_code    11375 non-null  int64  
 8   moo_name    11375 non-null  object 
 9   male        11375 non-null  int64  
 10  female      11375 non-null  int64  
 11  total       11375 non-null  int64  
 12  households  11375 non-null  int64  
 13  empty_drop  0 non-null      float64
dtypes: float64(1), int64(6), object(7)
memory usage: 1.2+ MB


,year_month,prov_code,prov_name,amp_code,amp_name,tam_code,tam_name,moo_code,moo_name,male,female,total,households,empty_drop
0,6612,0,ทั่วประเทศ,0,,0,,0,,32224008,33828607,66052615,28675857,NaN
1,6612,10,กรุงเทพมหานคร,0,,0,,0,,2555426,2916162,5471588,3251621,NaN
2,6612,10,กรุงเทพมหานคร,1001,ท้องถิ่นเขตพระนคร,0,,0,,19439,20815,40254,19042,NaN
3,6612,10,กรุงเทพมหานคร,1001,ท้องถิ่นเขตพระนคร,10010100,พระบรมมหาราชวัง,0,,1690,1206,2896,1195,NaN
4,6612,10,กรุงเทพมหานคร,1001,ท้องถิ่นเขตพระนคร,10010200,วังบูรพาภิรมย์,0,,4252,4237,8489,5326,NaN


### -- **Staging** --

In [47]:
df_dopa = df_dopa[df_dopa['prov_code']=='13']
(
    df_dopa
    .loc[lambda d: d['prov_code']=='13']
    .loc[lambda d: d['tam_code']!='0']
    .drop_duplicates(subset='tam_code')
)

,year_month,prov_code,prov_name,amp_code,amp_name,tam_code,tam_name,moo_code,moo_name,male,female,total,households,empty_drop
412,6612,13,จังหวัดปทุมธานี,1301,อำเภอเมืองปทุมธานี,13010200,ตำบลบ้านใหม่,0,,7229,8010,15239,8886,NaN
413,6612,13,จังหวัดปทุมธานี,1301,อำเภอเมืองปทุมธานี,13010400,ตำบลบ้านฉาง,0,,2710,2968,5678,2970,NaN
414,6612,13,จังหวัดปทุมธานี,1301,อำเภอเมืองปทุมธานี,13010500,ตำบลบ้านกระแชง,0,,1517,1614,3131,1150,NaN
415,6612,13,จังหวัดปทุมธานี,1301,อำเภอเมืองปทุมธานี,13010600,ตำบลบางขะแยง,0,,6908,7461,14369,7022,NaN
416,6612,13,จังหวัดปทุมธานี,1301,อำเภอเมืองปทุมธานี,13010700,ตำบลบางคูวัด,0,,14997,17271,32268,20842,NaN
417,6612,13,จังหวัดปทุมธานี,1301,อำเภอเมืองปทุมธานี,13010800,ตำบลบางหลวง,0,,3103,3364,6467,3371,NaN
418,6612,13,จังหวัดปทุมธานี,1301,อำเภอเมืองปทุมธานี,13010900,ตำบลบางเดื่อ,0,,7853,8821,16674,8039,NaN
419,6612,13,จังหวัดปทุมธานี,1301,อำเภอเมืองปทุมธานี,13011000,ตำบลบางพูด,0,,3304,3581,6885,2772,NaN
420,6612,13,จังหวัดปทุมธานี,1301,อำเภอเมืองปทุมธานี,13011300,ตำบลสวนพริกไทย,0,,5423,5859,11282,5596,NaN
421,6612,13,จังหวัดปทุมธานี,1301,อำเภอเมืองปทุมธานี,13011400,ตำบลหลักหก,0,,10545,11838,22383,12585,NaN
